# Reading the saved graph

`scripts/build_knowledge_graph.py` writes `data/kg/graph.json`, `mentions.jsonl`, `entities.jsonl`, and `eval.json`. This notebook reads those files. It does not retag the book.

Nodes are entities and documents. Edges include `REFERS_TO` (mention to entity), `NARRATED_BY`, `APPEARS_IN`, and `RELATION`. Phonograph documents are `RECORDED_BY` John Seward. `part_of` from Carfax to London is copied from config and marked `curated`.

`answer_queries` is the same function the tests call. Chapter keys on Mina's surnames are integers inside the function and strings in the JSON result, because JSON object keys are strings. Sort them as integers if you take a min or a max.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import json
from collections import Counter

from src.kg.entries import load_entries
from src.kg.evaluate import answer_queries
from src.kg.mentions import Mention

graph = json.loads((ROOT / "data" / "kg" / "graph.json").read_text(encoding="utf-8"))
mentions = [
    Mention(**json.loads(line))
    for line in (ROOT / "data" / "kg" / "mentions.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
entries = load_entries(ROOT / "data" / "dracula_ascii_rag.json")
answers = answer_queries(entries, mentions, graph)

print("nodes", len(graph["nodes"]), "edges", len(graph["edges"]))
print(Counter(edge["type"] for edge in graph["edges"]))
print("castle", answers["castle_id"], "count", answers["count_id"], "differ", answers["ids_differ"])
print("narrators", answers["narrators"])
print("documents", answers["document_count"])


## Mina's surname, the word Dracula, and Renfield

Mina's surfaced surname should move from Murray to Harker. The chapter numbers are strings in this dict. The first entry that contains the letters `dracula` can be earlier than the first entry where the person `count_dracula` appears, because the castle string also matches.

Seward's documents that also contain Renfield are the intersection of `NARRATED_BY john_seward` and `APPEARS_IN` from `rm_renfield`.


In [ ]:
surnames = answers["mina_surnames_by_chapter"]
for chapter in sorted(surnames, key=int):
    print(f"ch {int(chapter):2} {surnames[chapter]}")

print("first entry containing the string", answers["first_entry_string_dracula"])
print("first entry where the person appears", answers["first_entry_person_dracula"])
print("Seward entries that mention Renfield", answers["seward_entries_mentioning_renfield"])

relations = [edge for edge in graph["edges"] if edge["type"] == "RELATION"]
for edge in relations:
    print(edge["rel"], edge["source"], "->", edge["target"], edge.get("evidence"))
